Charger la data processed

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# charger la data processed
data_filled = pd.read_csv('../data/processed/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/chip_chain_log_returns.csv')

# 6 types d'indicateurs utiles

On souhaite avoir des indicateurs pour différentes parties des programmations à venir :
- HMM (Hidden Markov Model) afin d'identifier les différents états du marché
    - Bull market : faible volatilité, rendements positifs stables
    - Bear market / crise : très forte volatilité, krachs
    - Crab market : marché incertain
- LSTM (Long Short-Term Memory) pour prédire les mouvements
- Markowitz pour l'optimisation de portefeuille
- Pair trading pour l'arbitrage statistique

On créé un tableau pour stocker les 6 indicateurs

In [20]:
# créer le tableau pour les indicateurs personnalisés
features = pd.DataFrame(index=data_log_returns.index)

## Les 3 gros indicateurs

### Chip_Fear_Index La volatilité réalisée spécifique au secteur

On a déjà le VIX que l'on a télécharger qui concerne la peur globale du marché américain (S&P 500). Mais, il ne rend pas assez compte des mouvements du secteur : si une usine de TSMC brûle, le VIX va réagir trop peu par rapport à ce que l'on souhaite. Il nous faut un indice de peur sectoriel

Ainsi, on crée cela avec la volatilité réalisée (Realized Volatility), qui est l'écart-type glissant anualisé (252 jours), ici on prend une fenetre de 21 jours soit 1 mois 

In [21]:
# Créer Chip_Fear_Index : la volatilité réalisée, écart type des rendements log sur une fenêtre de 21 jours (1 mois) anualisée
numeric_df = data_log_returns.select_dtypes(include=[np.number])
volatility_21d = numeric_df.rolling(window=21).std() * np.sqrt(252) # ne pas inclure la colonne 'Date' dans le calcul de la volatilité
features['Chip_Fear_Index'] = volatility_21d[['ASML', 'TSM', '0981.HK', 'NVDA', 'AAPL']].mean(axis=1) # moyenne des volatilités des 5 actions, les plus importantes du secteur

features.shape

(3497, 1)

### Les Spreads (différentiels de rendements)

On sait qu'il existe des relations de corrélation voire une relation de reaction (c'est pourquoi le projet s'appelle ChipChainReaction)

#### Energie et matières brutes => fonderie => fabless => intégrateur (avec IA) (cf La Guerre des semi-conducteurs / Chip War)

Ainsi on souhaite mettre sur cette voie notre modèle pour qu'il comprenne ce lien, on créer donc des indiacteurs qui seront la soustraction entre deux rendements logs de tickers :
- Energie => fonderie (les couts de production)
- Fonderie => fabless (la capture des marges)
- Fabless => intégrateur (avec IA)

On ajoute aussi deux autres indicateurs de ce type :
- Le combat géopolitique entre Taiwan et la Chine : TSMC/SMIC
- Hardware contre software : Nvidia/Apple (Nvidia plus volatite que Apple)

Car $$ Spread = ln(\frac{P_{i,t}}{P_{i,t-1}}) - ln(\frac{P_{j,t}}{P_{j,t-1}}) = ln(\frac{P_{i,t}/P_{i,t-1}}{P_{j,t}/P_{j,t-1}}) = ln(\frac{P_{i,t}/P_{j,t}}{P_{i,t-1}/P_{j,t-1}}) =  ln(\frac{P_{i,t}}{P_{j,t}}) - ln(\frac{P_{i,t-1}}{P_{j,t-1}})$$

In [22]:
# 1. CRÉATION DES INDICES SECTORIELS (Paniers d'actions)
# Les matières dures (Énergie + Silicium)
features['Idx_Commodities'] = data_log_returns[['USO', 'PICK']].mean(axis=1)

# Les machines + L'usine (Les fondeurs occidentaux)
features['Idx_Western_Foundry'] = data_log_returns[['ASML', 'TSM']].mean(axis=1)

# Les concepteurs de puces (Les fabless)
features['Idx_Fabless'] = data_log_returns[['NVDA', 'AMD', 'AVGO', 'INTC']].mean(axis=1)

# Les géants de la Tech (Les acheteurs de puces)
features['Idx_Integrators'] = data_log_returns[['AAPL', 'MSFT', 'GOOGL', 'TSLA']].mean(axis=1)


# 2. CALCUL DES SPREADS MACRO (L'Effet Domino sur les secteurs entiers)
features['Spread_Macro_Commodity_Foundry'] = features['Idx_Commodities'] - features['Idx_Western_Foundry']
features['Spread_Macro_Foundry_Fabless'] = features['Idx_Western_Foundry'] - features['Idx_Fabless']
features['Spread_Macro_Fabless_Integrator'] = features['Idx_Fabless'] - features['Idx_Integrators']

# 3. LE SPREAD GÉOPOLITIQUE (Pur et isolé)
features['Spread_Geopolitics'] = data_log_returns['TSM'] - data_log_returns['0981.HK']

# 4. LE SPREAD ENTRE SOFTWARE ET HARDWARE
features['Spread_Software_Hardware'] = data_log_returns['NVDA'] - data_log_returns['AAPL']

features.shape

(3497, 10)

On a supprimé certains éléments dans les différents parties car ils sont très stables par rapport aux autres élémenrs (par exemple l'eau par rapport au pétrole)

### Les corrélations glissantes

Ici on calcule la correlation glissante c'est-à-dire $$ Corr(X_t,Y_t) = \frac{Cov(X_t,Y_t)}{\sigma_{X_t}*\sigma_{Y_t}} \quad sur\ une\ fenêtre\ mobile $$

Le but derrière est de l'exploiter pour surveiller les changements d'états (HMM), en effet, on retrouvera des corrélations pour des entreprises de même type mais lorsqu'on aura une cassure dans cette corrélation cela sera le signe d'une anomalie, une crise, des tensions géopolitiques, des problèmes de supply chain, ... Donc, en fonction de l'ampleur, un changement d'état

Le problème si on calcul toutes les corrélations glissantes : $$\binom{14}{2}=91 $$
On a :
- 91 nouvelles features (beaucoup trop)
- beaucoup de bruits
- de la redondance
- des risques d'overfitting
- un modèle plus lent
- des features sans sens

On doit donc trouver les relations importantes, c'est pourquoi j'ai choisi :
- Supply chain : ASML et TSM
- Géopolitique : TSM et 0981.HK
- Concurrence : NVDA et AMD
- Demande finale : NVDA et AAPL
- Infrastructure IA : MSFT et NVDA

In [23]:
features['Corr_ASML_TSM'] = data_log_returns['ASML'].rolling(window=63).corr(data_log_returns['TSM']) # 3 mois
features['Corr_TSM_0981'] = data_log_returns['TSM'].rolling(window=63).corr(data_log_returns['0981.HK'])
features['Corr_NVDA_AMD'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AMD'])
features['Corr_NVDA_AAPL'] = data_log_returns['NVDA'].rolling(window=63).corr(data_log_returns['AAPL'])
features['Corr_MSFT_NVDA'] = data_log_returns['MSFT'].rolling(window=63).corr(data_log_returns['NVDA'])
features.shape

(3497, 15)

## 3 autres types d'indicateurs pour détaillés au mieux les futurs modèles